# SMS Spam Filter – DLBDSME01 Model Engineering, Task 3

This notebook follows the CRISP-DM phases:
1. Business understanding
2. Data understanding
3. Data preparation
4. Modelling
5. Evaluation
6. Deployment (integration concept)

All figures, tables and metrics used in the case study are generated by this notebook.

In [1]:
# 0. Setup: imports, paths, logging and reproducibility settings
import hashlib
import logging
import platform
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
import sklearn

# --- Reproducibility --------------------------------------------------------
RANDOM_STATE = 42  # single seed used for every split and model in this notebook

# --- Project paths ----------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":  # VS Code runs the notebook from its own folder
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FILE = PROJECT_ROOT / "data" / "raw" / "SMSSpamCollection.csv"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"
LOGS_DIR = PROJECT_ROOT / "results" / "logs"

for folder in (FIGURES_DIR, TABLES_DIR, LOGS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# --- Logging (screen + file) ------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOGS_DIR / "notebook_run.log", mode="w", encoding="utf-8"),
    ],
    force=True,  # replace any handlers Jupyter created earlier
)
logger = logging.getLogger("spam_filter")

# --- Environment check ------------------------------------------------------
logger.info("Python       %s", platform.python_version())
logger.info("Interpreter  %s", sys.executable)
logger.info("numpy        %s", np.__version__)
logger.info("pandas       %s", pd.__version__)
logger.info("scikit-learn %s", sklearn.__version__)
logger.info("matplotlib   %s", matplotlib.__version__)
logger.info("Project root %s", PROJECT_ROOT)

# --- Raw data check ---------------------------------------------------------
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Raw data file not found: {DATA_FILE}")

sha256 = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
logger.info("Raw data     %s (%d bytes)", DATA_FILE.name, DATA_FILE.stat().st_size)
logger.info("SHA-256      %s", sha256)

2026-09-17 15:16:43 | INFO | Python       3.12.3
2026-09-17 15:16:43 | INFO | Interpreter  C:\AAAsantiago\IU Data Science\Semestres\6 Sexto semestre\Model Engineering\Code\IU_Model_Engineering\.venv\Scripts\python.exe
2026-09-17 15:16:43 | INFO | numpy        2.5.3
2026-09-17 15:16:43 | INFO | pandas       3.0.5
2026-09-17 15:16:43 | INFO | scikit-learn 1.9.1
2026-09-17 15:16:43 | INFO | matplotlib   3.11.2
2026-09-17 15:16:43 | INFO | Project root C:\AAAsantiago\IU Data Science\Semestres\6 Sexto semestre\Model Engineering\Code\IU_Model_Engineering
2026-09-17 15:16:43 | INFO | Raw data     SMSSpamCollection.csv (477907 bytes)
2026-09-17 15:16:43 | INFO | SHA-256      7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d


## 1. Business understanding

**Situation:** The company is opening a public short-message channel for customer feedback. Open channels attract spam.

**Goal:** A binary classification model that labels incoming short messages as spam or ham (legitimate).

**Service-team requirements (Task 3):**
- The filter is adjustable through spam-risk levels, from "low-risk" (very restrictive) to "high-risk" (not restrictive).
- Messages that pass the "low-risk" level are displayed immediately; all other messages are moved to a folder for further analysis.
- A strategy for handling that folder is needed so that legitimate customer messages are not lost.

**Implication for evaluation:** The two error types have different business consequences. A legitimate message classified as spam delays or loses customer feedback. A spam message classified as legitimate is displayed on the channel. The evaluation therefore reports both error types, not only overall accuracy.

## 2. Data understanding

### 2.1 Load the raw data and verify its structure

Loading decisions:
- The file is tab-separated and has no header row.
- Some messages contain quotation marks that belong to the text, so quote handling is switched off (`csv.QUOTE_NONE`).
- All values are read as text, and no text is converted into a missing value.

Checks: the number of loaded rows must equal the number of lines in the file, and the label counts must match the dataset documentation.

In [2]:
# 2.1 Load the raw data and verify its structure
import csv

COLUMN_NAMES = ["label", "message"]

# Message counts documented in the dataset description (Additional Information, section 3.3)
DOCUMENTED_COUNTS = {
    "ham": 450 + 3375 + 1002,  # Tagg PhD thesis + NUS SMS Corpus + SMS Spam Corpus v.0.1 Big
    "spam": 425 + 322,         # Grumbletext + SMS Spam Corpus v.0.1 Big
}

# --- Count non-empty lines in the raw file (one line = one message) ---------
with DATA_FILE.open(encoding="utf-8") as file:
    n_lines = sum(1 for line in file if line.strip())
logger.info("Non-empty lines in raw file: %d", n_lines)

# --- Load with explicit settings --------------------------------------------
sms_raw = pd.read_csv(
    DATA_FILE,
    sep="\t",                # tab-separated, despite the .csv extension
    header=None,             # no header row in the file
    names=COLUMN_NAMES,
    quoting=csv.QUOTE_NONE,  # quotation marks belong to the message text
    dtype=str,               # read everything as text
    keep_default_na=False,   # keep texts such as "NA" instead of turning them into missing values
    encoding="utf-8",
)
logger.info("Loaded with explicit settings: %d rows, %d columns", *sms_raw.shape)

# --- Comparison: what default quote handling would do -----------------------
sms_default = pd.read_csv(DATA_FILE, sep="\t", header=None, names=COLUMN_NAMES)
n_merged_rows = int(sms_default["message"].str.contains("\n", regex=False).sum())
logger.info(
    "Loaded with default quote handling: %d rows (%d messages lost, %d row(s) with merged lines)",
    len(sms_default), len(sms_raw) - len(sms_default), n_merged_rows,
)
del sms_default  # only needed for this comparison

# --- Structural checks ------------------------------------------------------
if len(sms_raw) != n_lines:
    raise ValueError(f"Row count {len(sms_raw)} does not match line count {n_lines}")

invalid_labels = set(sms_raw["label"]) - set(DOCUMENTED_COUNTS)
if invalid_labels:
    raise ValueError(f"Unexpected label values: {invalid_labels}")

label_counts = sms_raw["label"].value_counts()
for label, documented in DOCUMENTED_COUNTS.items():
    actual = int(label_counts[label])
    share = 100 * actual / len(sms_raw)
    logger.info("%-4s: %d messages (%.1f%%) | documented: %d", label, actual, share, documented)
    if actual != documented:
        raise ValueError(f"'{label}' count {actual} differs from documented {documented}")

logger.info("Structural checks passed: row count, label values and label counts match")
sms_raw.head()

2026-09-17 15:16:44 | INFO | Non-empty lines in raw file: 5574
2026-09-17 15:16:44 | INFO | Loaded with explicit settings: 5574 rows, 2 columns
2026-09-17 15:16:44 | INFO | Loaded with default quote handling: 5572 rows (2 messages lost, 1 row(s) with merged lines)
2026-09-17 15:16:44 | INFO | ham : 4827 messages (86.6%) | documented: 4827
2026-09-17 15:16:44 | INFO | spam: 747 messages (13.4%) | documented: 747
2026-09-17 15:16:44 | INFO | Structural checks passed: row count, label values and label counts match


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### 2.2 Data quality assessment

Each check counts the affected messages per class. Nothing is changed here; handling decisions follow in Data preparation.

- **Completeness:** empty messages
- **Uniqueness:** repeated messages
- **Consistency:** identical texts with different labels
- **Formatting and encoding:** whitespace, HTML escapes, placeholders, faulty characters

In [3]:
# 2.2 Data quality assessment (no data are changed here)
messages = sms_raw["message"]
n_per_label = sms_raw["label"].value_counts()


def count_flagged(dimension: str, check: str, flagged: pd.Series) -> dict:
    """Count flagged messages per label and as a share of each label."""
    counts = sms_raw.loc[flagged, "label"].value_counts()
    ham, spam = int(counts.get("ham", 0)), int(counts.get("spam", 0))
    return {
        "Dimension": dimension,
        "Check": check,
        "Ham": ham,
        "Spam": spam,
        "Total": ham + spam,
        "% of ham": round(100 * ham / n_per_label["ham"], 1),
        "% of spam": round(100 * spam / n_per_label["spam"], 1),
    }


quality_checks = [
    count_flagged("Completeness", "Empty or whitespace-only message",
                  messages.str.strip() == ""),
    count_flagged("Uniqueness", "Duplicate of an earlier message (same label and text)",
                  sms_raw.duplicated()),
    count_flagged("Consistency", "Same text with conflicting labels",
                  sms_raw.groupby("message")["label"].transform("nunique") > 1),
    count_flagged("Formatting", "Leading or trailing whitespace",
                  messages != messages.str.strip()),
    count_flagged("Formatting", "HTML-escaped characters (&lt; &gt; &amp;)",
                  messages.str.contains(r"&(?:lt|gt|amp);", regex=True)),
    count_flagged("Formatting", "Placeholder instead of original content (e.g. &lt;#&gt;, &lt;TIME&gt;)",
                  messages.str.contains(r"&lt;(?:#|[A-Z]+)&gt;", regex=True)),
    count_flagged("Encoding", "Windows control character instead of punctuation (e.g. \\x92)",
                  messages.str.contains(r"[\x80-\x9f]", regex=True)),
]
quality_table = pd.DataFrame(quality_checks)
quality_table.to_csv(TABLES_DIR / "tab_data_quality.csv", index=False)

# --- Class balance before and after removing duplicates ---------------------
unique_counts = sms_raw.drop_duplicates()["label"].value_counts()
logger.info("All messages:    ham %d, spam %d (spam share %.1f%%)",
            n_per_label["ham"], n_per_label["spam"], 100 * n_per_label["spam"] / len(sms_raw))
logger.info("Unique messages: ham %d, spam %d (spam share %.1f%%)",
            unique_counts["ham"], unique_counts["spam"], 100 * unique_counts["spam"] / unique_counts.sum())
logger.info("Saved %s", "tab_data_quality.csv")
quality_table

2026-09-17 15:19:38 | INFO | All messages:    ham 4827, spam 747 (spam share 13.4%)
2026-09-17 15:19:38 | INFO | Unique messages: ham 4518, spam 653 (spam share 12.6%)
2026-09-17 15:19:38 | INFO | Saved tab_data_quality.csv


,Dimension,Check,Ham,Spam,Total,% of ham,% of spam
0,Completeness,Empty or whitespace-only message,0,0,0,0.0,0.0
1,Uniqueness,Duplicate of an earlier message (same label an...,309,94,403,6.4,12.6
2,Consistency,Same text with conflicting labels,0,0,0,0.0,0.0
3,Formatting,Leading or trailing whitespace,156,31,187,3.2,4.1
4,Formatting,HTML-escaped characters (&lt; &gt; &amp;),309,0,309,6.4,0.0
5,Formatting,Placeholder instead of original content (e.g. ...,236,0,236,4.9,0.0
6,Encoding,Windows control character instead of punctuati...,33,2,35,0.7,0.3


### 2.3 Message length per class

Length in characters and words, measured on the raw text.

In [4]:
# 2.3 Message length per class
length_data = pd.DataFrame({
    "label": sms_raw["label"],
    "characters": messages.str.len(),
    "words": messages.str.split().str.len(),
})

length_table = (
    length_data.groupby("label")[["characters", "words"]]
    .agg(["median", "mean", "min", "max"])
    .round(1)
)
length_table.columns = [f"{stat.capitalize()} {measure}" for measure, stat in length_table.columns]
length_table.to_csv(TABLES_DIR / "tab_message_length.csv")
logger.info("Saved %s", "tab_message_length.csv")
length_table

2026-09-17 15:23:43 | INFO | Saved tab_message_length.csv


,Median characters,Mean characters,Min characters,Max characters,Median words,Mean words,Min words,Max words
label,,,,,,,,
ham,52.0,71.5,2,910,11.0,14.3,1,171
spam,149.0,138.7,13,223,25.0,23.9,2,35
